# Extracción y Análisis de Comentarios de Mercado Libre

Este notebook realiza el scraping de comentarios de un producto específico en Mercado Libre, procesa el texto y realiza un análisis de sentimiento básico junto con visualizaciones.

## 1. Carga de librerías

Cargamos las librerías necesarias para scraping, procesamiento de lenguaje natural y visualización.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re
from collections import Counter
from urllib.parse import urljoin

# Descargar recursos necesarios de NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

## 2. Descargar datos de Mercado Libre

Extraemos los comentarios y sus calificaciones utilizando Web Scraping. Navegamos al enlace de "Mostrar todas las opiniones" para obtener el total de comentarios (aprox. 23).

In [ ]:
base_url = "https://www.mercadolibre.com.mx/sillon-puff-sofa-de-suelo-con-espuma-memoria-y-respaldo-color-verde-musgo/p/MLM61732880#reviews"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}

session = requests.Session()
comentarios_raw = []

def extraer_de_pagina(url_objetivo):
    print(f"Extrayendo de: {url_objetivo}")
    res = session.get(url_objetivo, headers=headers)
    s = BeautifulSoup(res.text, 'html.parser')
    articulos = s.find_all('article', class_='ui-review-capability-comments__comment')
    
    for art in articulos:
        rating_element = art.find('p', class_='andes-visually-hidden')
        rating_text = rating_element.get_text() if rating_element else "0"
        rating = int(re.search(r'\d+', rating_text).group()) if re.search(r'\d+', rating_text) else 0
        
        content_element = art.find('p', class_='ui-review-capability-comments__comment__content')
        content = content_element.get_text().strip() if content_element else ""
        
        if content:
            comentarios_raw.append({'rating': rating, 'comentario': content})

# 1. Obtener página principal para encontrar el enlace a todas las reseñas
response = session.get(base_url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# 2. Buscar el enlace que contiene '/noindex/catalog/reviews/'
link_element = soup.find('a', href=re.compile(r'/noindex/catalog/reviews/'))

if link_element:
    reviews_base_url = urljoin(base_url, link_element.get('href'))
    # Mercado Libre suele usar paginación por 'offset'
    # Extraemos la primera página (0-15) y la segunda (15-30) para asegurar los 23 comentarios
    for offset in [0, 15]:
        url_con_offset = f"{reviews_base_url}&offset={offset}"
        extraer_de_pagina(url_con_offset)
else:
    print("No se encontró el enlace de reseñas completas, extrayendo de la página principal.")
    extraer_de_pagina(base_url)

df = pd.DataFrame(comentarios_raw).drop_duplicates(subset=['comentario'])
print(f"Se extrajeron {len(df)} comentarios únicos.")
df.head()

## 3. Clasificación de comentarios

Clasificamos los comentarios basándonos en la calificación de estrellas (4-5: Positivo, 1-2: Negativo, 3: Neutro).

In [ ]:
def clasificar_sentimiento(rating):
    if rating >= 4: return 'Positivo'
    elif rating <= 2: return 'Negativo'
    else: return 'Neutro'

df['sentimiento'] = df['rating'].apply(clasificar_sentimiento)
df.head()

## 4. Limpieza de datos

Eliminamos caracteres especiales y convertimos a minúsculas para normalizar el texto.

In [ ]:
def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúñ\s]', '', texto)
    return texto

df['limpio'] = df['comentario'].apply(limpiar_texto)
df.head()

## 5. Tokenización y Eliminación de Stopwords

Dividimos el texto en palabras individuales y eliminamos las palabras comunes (conectores).

In [ ]:
stop_words = set(stopwords.words('spanish'))

def procesar_tokens(texto):
    tokens = word_tokenize(texto)
    tokens_limpios = [w for w in tokens if w not in stop_words and len(w) > 2]
    return tokens_limpios

df['tokens'] = df['limpio'].apply(procesar_tokens)
df.head()

## ANÁLISIS 1 - Distribución de Sentimientos

Visualización de cuántos comentarios son positivos, negativos o neutros.

In [ ]:
counts = df['sentimiento'].value_counts()
plt.figure(figsize=(8, 5))
counts.plot(kind='bar', color=['green', 'red', 'gray'])
plt.title('Distribución de Sentimientos')
plt.xlabel('Sentimiento')
plt.ylabel('Cantidad')
plt.show()

## ANÁLISIS 2 - Palabras Más Frecuentes

Identificamos los términos que más aparecen en el total de comentarios.

In [ ]:
todas_palabras = [palabra for lista in df['tokens'] for palabra in lista]
conteo_palabras = Counter(todas_palabras).most_common(15)
df_frecuencia = pd.DataFrame(conteo_palabras, columns=['Palabra', 'Frecuencia'])

plt.figure(figsize=(10, 6))
plt.barh(df_frecuencia['Palabra'], df_frecuencia['Frecuencia'], color='skyblue')
plt.title('Top 15 Palabras más frecuentes')
plt.gca().invert_yaxis()
plt.show()

## ANÁLISIS 3 - Comparativa de Palabras por Sentimiento

Listamos los términos más usados en comentarios positivos vs negativos.

In [ ]:
pos_palabras = [p for lista in df[df['sentimiento']=='Positivo']['tokens'] for p in lista]
neg_palabras = [p for lista in df[df['sentimiento']=='Negativo']['tokens'] for p in lista]

top_pos = Counter(pos_palabras).most_common(10)
top_neg = Counter(neg_palabras).most_common(10)

print("Top 10 palabras en Comentarios POSITIVOS:", top_pos)
print("Top 10 palabras en Comentarios NEGATIVOS:", top_neg)

## ANÁLISIS 4 - Nube de Palabras

Representación visual de los términos predominantes según el sentimiento.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

if pos_palabras:
    wc_pos = WordCloud(width=400, height=400, background_color='white').generate(" ".join(pos_palabras))
    ax1.imshow(wc_pos)
    ax1.set_title('Nube de Palabras - Positivas')
    ax1.axis('off')

if neg_palabras:
    wc_neg = WordCloud(width=400, height=400, background_color='black', colormap='Reds').generate(" ".join(neg_palabras))
    ax2.imshow(wc_neg)
    ax2.set_title('Nube de Palabras - Negativas')
    ax2.axis('off')

plt.show()

## Conclusión Final

Resumen de los hallazgos principales basados en los datos extraídos y analizados.